# JUnit Behavior Tests

In this lesson, you will learn to write and run automated checks for a Java method's successful results, boundary values, and required exceptions.

CSC-239 · Module 7 · Lesson 4 of 4

You have compared printed results with predictions. Now you will turn known expectations into repeatable checks using JUnit. You will also investigate a misleading passing test and confirm that repaired checks detect an incorrect implementation.

Use the [Module 7 glossary](terms.md) to revisit the terms after their explanations.

## Learning Goals

- Write independent JUnit Jupiter tests for normal, boundary, and required-exception behavior.
- Run a selected notebook-defined test class and compare its result counts with the intended tests.
- Diagnose a false pass and use unchanged behavior expectations to verify a production-code repair.

## Why This Matters

A change to a useful method can accidentally break behavior that previously worked. A **regression** is that loss of working behavior after a change. Repeatable tests let a development team check important cases again without relying on someone to remember and manually inspect every result.

Tests are useful only when their expectations come from the requirements. A test that repeats the implementation's mistake, or never runs its important check, can report success while the application is wrong. You will learn to examine both what a test checks and whether the intended tests actually ran.

This lesson brings together method contracts, boundary reasoning, exceptions, and automatic session cleanup. The same habits will help you check file-processing code and larger applications later in the course.

## Check Your Starting Point

A campus workshop has eight seats. The supplied method receives the seat capacity and the number of reservations. It rejects a request that reserves more seats than the room holds; otherwise, it returns the seats remaining.

Read the method and consider two separate calls: `remaining(8, 3)` and `remaining(8, 9)`. For each call, predict whether it returns a value or raises an exception. State the returned value or the exception type and message. Explain which statement determines each result. You can use your existing method and exception knowledge; no JUnit knowledge is needed yet.

```java
class StartingCheck {
    public static int remaining(int capacity, int reserved) {
        if (reserved > capacity) {
            throw new IllegalArgumentException("Too many reservations.");
        }
        return capacity - reserved;
    }
}
```

In [ ]:
Your response:

remaining(8, 3): predicted behavior and deciding statement:

remaining(8, 9): predicted behavior and deciding statement:

What a matching caller handler would receive if an exception occurs:


<details><summary>Show answer: follow the method contract</summary>

```text
remaining(8, 3): returns 5
remaining(8, 9): raises IllegalArgumentException with message Too many reservations.
```

Three reservations fit within eight seats, so the first call returns five. Nine reservations exceed capacity, so the second executes throw and never reaches return. The expected failure is part of this small method contract. The lesson will teach how to turn these expectations into automated checks.

</details>

## Video Demonstration

Watch how the demonstration checks a normal result, a zero boundary, and required invalid-input behavior. The reading below explains each JUnit operation before you use it in practice.

The video uses the [JUnit video starter project](junit_video_starter/README.md) in the Java Workspace. Open `junit_video_starter` as the Workspace folder so its **Run Code** settings apply. It includes an empty `Main.java` and the JUnit library. The video builds three tests. The first guided practice example includes a fourth test, for zero kits. The notebook cells use the dependency setup explained below.

<video controls preload="metadata" width="960">
  <source src="media/04_junit_behavior_tests/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_junit_behavior_tests/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the junit behavior tests demonstration transcript](media/04_junit_behavior_tests/transcript.md).

## Concept

### Turn a requirement into a small behavior check

A workshop coordinator needs labels for groups of supply kits. Each group receives the same number of kits, and each kit needs one label. For nonnegative counts, the required label count is the number of groups multiplied by the kits per group. Zero groups or zero kits requires zero labels. A negative count must raise `IllegalArgumentException` with the message `Counts must be nonnegative.`

A **unit test** is an automated check of a small behavior using controlled inputs and an expected outcome. Two groups receiving three kits each need six labels. We can choose that expectation from the scenario before looking at the method's implementation. This gives the test an independent basis for detecting an incorrect calculation.

JUnit is a Java testing framework that provides test annotations, assertions, and tools for running tests. We will add its library, define a few checks, and use a supplied runner to collect their results.

A **boundary case** checks the edge of an allowed range. Zero is the smallest allowed group or kit count here, so each zero-input case checks a boundary. An **invalid-input case** checks a value the contract rejects, such as a negative count. We need both kinds of cases because accepting zero and rejecting negatives are different requirements.

### Add the library before importing its classes

An **external dependency** is a library supplied in addition to Java's standard library. This lesson uses JUnit Jupiter 5.13.4 with JUnit Platform 1.13.4. One pinned standalone **JAR**, a Java archive containing compiled classes and resources, supplies the components for these examples.

The IJava setup line is:

```text
%maven org.junit.platform:junit-platform-console-standalone:1.13.4
```

This is an IJava kernel command, not a Java statement for a `.java` file. Its **dependency coordinate** identifies a group, an artifact, and a version, separated by colons. The kernel obtains the archive and makes its classes available on the **classpath**, the locations Java uses to find classes.

An `import` later lets Java source use a class's short name. It does not download the library. Run the visible setup cell once in each fresh kernel before running any JUnit example. The first download needs access to Maven Central; a later run may use a cached archive. Setup normally prints nothing. If it reports an error, resolve that dependency problem before interpreting later results as behavior-test failures.

### Load JUnit for the examples

Run this dependency cell before the complete Java programs. It belongs to the notebook kernel; do not copy it into a Java class. If the download reports an error, address that setup problem before continuing.

In [ ]:
%maven org.junit.platform:junit-platform-console-standalone:1.13.4


### Mark a method as a test

The JUnit annotation **`@Test`** identifies a method that the testing framework should discover and run. It is an annotation like `@Override`, but its job is different: `@Override` checks an inherited method relationship, while `@Test` marks a test for JUnit.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
class SumTest {
    public SumTest() { }
    @Test
    public void addsTwoCounts() {
        Assertions.assertEquals(5, 2 + 3);
    }
}
```

This is a reading example; defining it alone does not produce a test report. The two `import` statements make the JUnit class names available in short form after dependency setup. The keyword **`class`** groups the test methods in `SumTest`. The explicit empty constructor follows the class pattern you already know. Its name matches the class, and its empty body performs no additional setup.

The keyword **`public`** makes the method accessible. The keyword **`void`** says the method returns no value: its assertion reports whether the check succeeds. Our test methods are instance methods, so they omit **`static`**. Their empty parameter lists mean JUnit can call them without test arguments in this example. Each supplies its own inputs rather than depending on another test to prepare a value.

Defining this class does not run `addsTwoCounts`. It only makes the class and its annotated method available. The runner introduced below performs discovery and execution.

### Compare an expected result with the actual result

A **test assertion** checks a condition and reports a test failure when it is not met. In `Assertions.assertEquals(expected, actual)`, the first argument is the expected value and the second is the observed value.

```java
import org.junit.jupiter.api.Assertions;
int actual = 2 * 3;
Assertions.assertEquals(6, actual);
System.out.println("Check passed");
```

The import supplies the short name `Assertions`; the library must already be loaded. This reading example prints `Check passed`. The assertion itself normally prints nothing on success. If the expected value were incorrectly changed to `7`, the assertion would fail before the print statement. When the same assertion is inside a JUnit test, the runner records that failure in the test results.

For a method test, the actual argument calls the production method, such as `LabelTools.labelCount(2, 3)`. **Production code** is the application code being tested. Choose the expected `6` from the stated label requirement; do not compute both sides by copying the same expression from the implementation. Otherwise, the same mistake can appear on both sides and escape detection.

### Require the failure when input is invalid

For negative groups, a returned number is not a successful application result. The contract requires an exception. That expected exception can therefore make a test succeed:

```java
try {
    LabelTools.labelCount(-1, 3);
    Assertions.fail("Expected IllegalArgumentException");
} catch (IllegalArgumentException problem) {
    Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
}
```

This fragment belongs inside a test method. First, it calls the production method with invalid input. If the method raises the required exception, control moves to the specific handler, which checks the required message. The following `Assertions.fail` call is skipped because the application call did not return normally.

If the production method incorrectly returns a number, execution reaches **`Assertions.fail`**, which deliberately reports a failed assertion. A JUnit assertion failure is a different type of failure from `IllegalArgumentException`, so this handler does not swallow it.

That extra line closes a testing gap. A handler-only test can do nothing when no exception occurs and still pass. The explicit failure assertion ensures that an unexpected normal return is detected. Later Java topics provide other ways to express exception tests; this pattern uses the control flow you have already learned.

The diagram follows the two paths through the exception test. Its controlled comparison changes the guard and the failure assertion one at a time.

<details class="animation-panel" open>
<summary>Detect a missing required exception — show or hide animation</summary>
<p><img src="media/04_junit_behavior_tests/exception_test_false_positive.gif" alt="With a missing guard, labelCount(-1, 3) returns -3. A catch-only test misses the defect. Adding Assertions.fail detects the normal return. Restoring the guard transfers control to the handler, where the message check succeeds." width="960" style="max-width:100%;height:auto;"></p>
</details>

When the application returns normally, its catch is skipped. The failure assertion after the call closes that gap. With the guard restored, the required exception transfers control to the matching handler, which checks the message. The required behavior stays the same throughout the comparison. This silent loop lasts 13 seconds. Hide it with the summary control when you want to focus on the reading. [View the still image for this sequence](media/04_junit_behavior_tests/exception_test_false_positive_still.png).


### Select the class and execute its tests

A **test runner** discovers tests, executes them, and collects their results. In a larger project, an editor or build tool often starts this process. This notebook provides a Launcher scaffold, so you can focus on writing behavior checks without constructing a separate project.

A **class literal** such as `LabelToolsTest.class` refers to the class itself. It is not a string containing a name, and it does not construct a test object. Passing it to `DiscoverySelectors.selectClass` identifies the current notebook-defined class to test.

Read the supplied runner in five steps:

1. `LauncherDiscoveryRequestBuilder.request()` starts a request. `selectors(...)` identifies the test class, and `build()` finishes the request. These chained calls use returned objects for the next operation.
2. `new SummaryGeneratingListener()` creates the **summary listener**, the object that collects test results. A listener is an object notified as testing progresses; you use this supplied implementation rather than write your own.
3. `LauncherFactory.openSession()` opens a session inside try-with-resources. The cleanup pattern from the previous lesson closes the session when its body ends.
4. `getLauncher()` provides the launcher. `registerTestExecutionListeners(listener)` connects the collector, and `execute(request)` runs the selected tests.
5. `listener.getSummary()` provides the results. The final statements print the counts of succeeded and failed tests.

Keep the complete supplied runner with each test program. After editing a test class, rerun that class definition and its runner so the selection refers to the current definition. A kernel restart also requires the dependency setup again.

These are excerpts from the complete program below, for reading in the order shown. First, the request identifies which class contains the tests:

```java
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
```

The declaration stores the finished request in `request`. The indented line continues the same statement; it does not start testing. The next declaration constructs a fresh collector in `listener`. A fresh collector prevents an earlier run's summary from serving as the report for this run.

```java
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
```

Inside the session, the first statement stores its launcher. The second connects the collector before testing starts, so the collector receives this run's results. The third executes the selected request. When the body ends, try-with-resources closes the session. The program can then read the completed summary from the listener. Keep this supplied structure intact while changing the test methods.

### Check that the expected tests ran

A Java cell can finish normally while JUnit reports `Failed: 1`. The runner records an assertion failure as a test result; normal completion of the surrounding cell does not turn that failed test into a pass.

The three-test worked example requires:

```text
Succeeded: 3
Failed: 0
```

Zero failed tests alone is insufficient. A run that discovered no tests would not prove any of the three behaviors. Compare both counts with the intended number of tests, and investigate an unexpected total instead of calling it success.

These examples have no intentionally skipped or aborted tests. In this limited setting, succeeded plus failed should equal the intended count. JUnit also tracks skipped, aborted, and other execution states, so that sum is not a universal definition of every possible test outcome.

The diagram connects the class definition with the report for the three-test label-count example. The displayed phases describe the runner’s work; the list of test methods does not prescribe their execution order.

<details class="animation-panel" open>
<summary>Follow the three-test execution cycle — show or hide animation</summary>
<p><img src="media/04_junit_behavior_tests/test_execution_cycle.gif" alt="LabelToolsTest is defined, selected by its class literal, and discovered as three test methods. The runner checks the normal, zero, and required-exception behaviors and reports Succeeded: 3 and Failed: 0." width="960" style="max-width:100%;height:auto;"></p>
</details>

Defining the class makes tests available. Selection identifies that class; execution evaluates its checks and sends results to the listener. The final report must agree with both the required behavior and the intended number of tests. These three tests use independent inputs, so their execution order does not affect the expectations. This silent loop lasts 13 seconds. Hide it with the summary control when you want to focus on the reading. [View the still image for this sequence](media/04_junit_behavior_tests/test_execution_cycle_still.png).


### Check assertions and preserve behavior

An **assertion sanity check** deliberately changes a known correct expected value to an incorrect one, then confirms that the test reports failure. For example, changing expected `6` to `7` should fail the positive-count test even though the application still correctly returns `6`. Restore the correct expectation afterward. This checks the assertion path; it is not itself a production regression.

A **regression test** preserves an application's required behavior across changes. If a production edit removes the negative-input guard, the unchanged exception test should detect that the method now returns instead of throwing. Restoring the guard should make that same test pass again.

Keep the two experiments distinct. One checks whether an assertion reacts to a mismatch. The other checks whether a production change broke required behavior. Neither proves coverage of every possible defect, which is why the practice adds separate boundary and invalid-input cases.

This comparison uses the original three-test suite. Watch the production guard change while the test expectations stay fixed. The final state labels the wrong-expected-value experiment separately.

<details class="animation-panel" open>
<summary>Keep the expected rule while repairing the application — show or hide animation</summary>
<p><img src="media/04_junit_behavior_tests/regression_and_repair.gif" alt="The original suite reports three successes. Removing the negative-count guard makes negativeGroups fail, leaving two successes and one failure. Restoring the guard restores three successes. A separate edit changes expected six to seven, making positiveCounts fail without changing the application." width="960" style="max-width:100%;height:auto;"></p>
</details>

Removing the guard breaks the required rejection behavior. The unchanged complete exception test detects that regression. Restoring the guard restores the three-test report to three successes and zero failures. Deliberately changing expected 6 to 7 instead tests the assertion’s ability to report a mismatch; restore 6 after that separate experiment. This silent loop lasts 13 seconds. Hide it with the summary control when you want to focus on the reading. [View the still image for this sequence](media/04_junit_behavior_tests/regression_and_repair_still.png).


## Worked Example

### Step 1: implement the label-count contract

The workshop needs one label for every kit. `LabelTools.labelCount` receives the number of groups and the kits in each group. These parameter names keep the two counts distinct even though both are integers.

Read the method's decision before running the complete program:

```java
if (groups < 0 || kits < 0) {
    throw new IllegalArgumentException("Counts must be nonnegative.");
}
return groups * kits;
```

The `if` checks the rule before doing the calculation. The conditional-or operator `||` makes the condition true when either count is negative; Java only evaluates its right condition when the left is false. The **`throw`** statement raises the required exception and ends this call's normal path. With two nonnegative counts, execution reaches **`return`**, which supplies the product to the caller. Zero passes the guard and produces zero labels.

### Step 2: check three distinct behaviors

The first test uses two groups with three kits in each. We can count six labels from that requirement without borrowing the application's calculation:

```java
@Test
public void positiveCounts() {
    Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
}
```

`@Test` marks this method for discovery. Its descriptive name records the behavior being checked. Inside the method, the first assertion argument supplies expected `6`; Java evaluates the application call for the second argument to obtain the actual count. The comparison succeeds only when the values agree.

The `zeroGroups` test applies the same pattern to no groups and expects zero. The `negativeGroups` test uses the exception pattern just explained: require the exception, detect an unexpected return, and check the message in the specific handler. Each test supplies its own inputs and does not need another test to run first.

### Step 3: run the selected class and inspect both counts

The supplied runner selects `LabelToolsTest.class`, connects a new summary listener, and executes the request. Try-with-resources closes the runner session before the two summary prints. The first print reports how many checks succeeded; the second reports how many failed.

The following complete program assembles the imports, application method, three tests, and runner. Run it after the dependency setup and compare the result with the three intended tests.

In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


The result is:

```text
Succeeded: 3
Failed: 0
```

The normal product and zero boundary match their expected values. The negative-input check also succeeds: it required an exception, received that exception, and confirmed the message. Success here means the observed behavior matched the contract; it does not mean every production call returned a number.

All three intended checks passed. That is stronger evidence than seeing zero failures alone, but it is still limited to the behaviors checked. The practice adds a zero-kits boundary, exposes an incomplete exception test, and separates assertion checks from repairing production code.

## Guided Practice

### Predict the added boundary test

The next complete program adds `zeroKits` to the worked label-count example. Read it without running it. Count the methods marked `@Test` in the class selected by `LabelToolsTest.class`. For each test, identify the input and the behavior it requires, then predict whether the check will succeed.

Write both predicted summary lines. Explain how the added boundary affects the number of executed tests and why an expected application exception can be a successful test outcome. Keep this prediction before running the program or opening feedback.

In [ ]:
Your response:

Number of selected @Test methods:

positiveCounts: inputs, required behavior, and predicted test result:

zeroGroups: inputs, required behavior, and predicted test result:

negativeGroups: inputs, required behavior, and predicted test result:

zeroKits: inputs, required behavior, and predicted test result:

Predicted Succeeded line:

Predicted Failed line:

Why an expected application exception can count as success:

How the new boundary changes the total test count:


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


### Compare the actual test report

Run the complete program after the dependency setup has succeeded in this Java kernel. Preserve your prediction and record both actual summary lines. Add their counts and compare the total with the number of tests you intended to run. Explain any difference between your prediction and the report.

Also distinguish the dependency setup from the imports in the Java cell. State what you must rerun after restarting the kernel so the same test program can execute again.

In [ ]:
Your response:

Original predicted Succeeded and Failed counts:

Actual Succeeded and Failed lines:

Intended test count and actual sum of the counts:

Explanation of agreement or any mismatch:

What dependency setup provides versus what imports do:

What to rerun after restarting the kernel:


### Connect each check to the runner

Use the four-test program you just ran. For each test method, record its inputs and the expected behavior chosen from the label-count contract. Explain how you can choose that expectation without copying the implementation expression.

Identify the class literal that selects the tests. Then trace request creation and class selection, listener creation and registration, execution, automatic session closure, and summary reading in the order they occur. Explain why defining the test class alone does not run it and why these tests do not need a particular execution order.

In [ ]:
Your response:

positiveCounts: inputs and independently chosen expected behavior:

zeroGroups: inputs and independently chosen expected behavior:

negativeGroups: inputs and independently chosen expected behavior:

zeroKits: inputs and independently chosen expected behavior:

Class literal and selected class:

Runner operations in order:

Session closure relative to summary printing:

Dependency setup, imports, class definition, and execution: distinct jobs:

Why these tests are independent of execution order:


<details>
<summary>Show answer</summary>

The selected LabelToolsTest class has four @Test methods. positiveCounts checks the known product 2 times 3 against 6. zeroGroups checks that zero groups produce zero labels. negativeGroups succeeds because the application raises IllegalArgumentException with the required message; the line that would report a missing exception is skipped. zeroKits checks the separate zero-kit boundary with groups still positive. All four agree with the production contract, so the runner reports Succeeded: 4 and Failed: 0. The class literal selects the test class; defining the class alone would not execute its tests. The new listener receives this run's counts, and the runner session closes after execute finishes. The expected values come from the behavior contract, not from duplicating the implementation expression.

The setup cell loads the pinned JUnit archive into the current kernel. Every full Java program supplies all imports, its application/test definitions and its own runner, but the external library still must be available in that kernel. A normally completed kernel cell does not by itself prove these test expectations passed; the printed counts provide the relevant evidence.

The request selects LabelToolsTest.class. A new SummaryGeneratingListener collects results only after it is registered with the launcher. execute runs the request, and the try-with-resources statement closes the session before the two summary prints. The setup supplies JUnit classes; imports shorten their names; @Test marks the test methods; the explicit runner discovers and executes them. Each test makes its own calls with local inputs and does not rely on another test running first.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Counting the expected application exception as a failed test. Counting ordinary helper methods as discovered @Test methods. Reusing the worked example's result count after adding another test.

</details>

### Predict what an incomplete test reports

The next program deliberately removes the negative-input guard from `labelCount`. It also removes the `Assertions.fail` call from `negativeGroups`. The other checks are unchanged. Read this version without running it.

Trace `labelCount(-1, 3)`: determine what it returns and whether the specific handler executes. Decide whether the test still checks the requirement that negative counts must be rejected. Predict both summary lines before running the program.

In [ ]:
Your response:

Value or exception produced by labelCount(-1, 3):

Whether its handler executes, and why:

Requirement the test actually checks on this path:

Predicted Succeeded line:

Predicted Failed line:


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


### Judge the report against the requirement

Run the first modified program and record its actual summary. Compare the report with your prediction and with the requirement to reject negative counts. Explain whether this report provides evidence that the negative-input rule works. Identify the unchecked path inside `negativeGroups`.

In [ ]:
Your response:

Original predicted counts:

Actual Succeeded and Failed lines:

Does the method satisfy the negative-input rule? Evidence:

Path that the test leaves unchecked:

What this report does and does not establish:


### Predict the effect of restoring the missing check

The next program restores `Assertions.fail` immediately after the negative-input call. The application still has no negative-input guard. Read this version before running it.

Trace the statement reached if `labelCount` returns normally. Explain whether `catch (IllegalArgumentException problem)` handles the failure reported by that assertion. Predict both test counts, identifying which test your reasoning affects.

In [ ]:
Your response:

Statement reached after the application returns normally:

Whether the specific catch handles the assertion failure, and why:

Affected test method:

Predicted Succeeded and Failed lines:


In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


### Compare detection with repaired behavior

Run the second modified program and record its summary. Explain what the result tells you about the test and the application separately. Then rerun the original complete four-test program from the earlier boundary prediction. That program includes both the application guard and the complete exception check. Record its summary too.

Compare the original program with the first modified version. Explain why matching passing counts would not, by themselves, prove that both versions enforce the same rule.

In [ ]:
Your response:

Second modified program: actual Succeeded and Failed lines:

What the test detected in the application:

Original four-test program after restoration: actual counts:

Why the restored passing result has stronger support than the first modified report:

Why normal kernel-cell completion alone does not establish that all tests passed:


<details>
<summary>Show answer</summary>

Both supplied comparison programs deliberately remove the production guard, so labelCount(-1, 3) incorrectly returns -3. In the first, negativeGroups also lacks Assertions.fail. The application call returns normally, the catch is skipped, and the test reaches its end without reporting the missing exception. The three numeric tests still pass, so the summary falsely looks good at 4 successes and 0 failures.

Restoring Assertions.fail while leaving the production defect in place makes negativeGroups report the missing exception; the summary becomes 3 successes and 1 failure. The catch handles only IllegalArgumentException and therefore does not hide the JUnit assertion failure. The Launcher still completes normally in both cases.

To restore the application contract, put back the negative-input guard as well as retaining the complete exception test; the correct four-test program then reports 4 successes and 0 failures.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Treating the first zero-failure count as proof that invalid inputs are rejected. Removing Assertions.fail because a catch is already present. Repairing only the expected test count while leaving the production defect.

**Check case 2.** The production call returns when it should throw. Assertions.fail records a failed test, and the specific IllegalArgumentException catch does not intercept the assertion failure.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 1
```

**Check case 3.** The negative-input call now raises the required application exception with the correct message, and the full test checks it. All four tests then agree with the stated contract.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

</details>

### Complete the test and runner connections

The draft below is intentionally incomplete and is for reading until you repair it. Plan a replacement for each placeholder using `Test`, `assertEquals`, `fail`, and `LabelToolsTest`, each once. Explain each replacement’s job. Identify which argument of the equality assertion is the independently chosen expectation and which is the actual result. Predict both summary lines.

Then copy the draft into the empty work cell and make only these four replacements. Keep the dependency setup in its own cell.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @TEST_MARK
    public void positiveCounts() {
        Assertions.CHECK_EQUAL(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.REPORT_MISSING_EXCEPTION("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(TEST_CLASS.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

In [ ]:
Your response:

TEST_MARK replacement and job:

CHECK_EQUAL replacement and job:

REPORT_MISSING_EXCEPTION replacement and job:

TEST_CLASS replacement and job:

Expected argument versus actual argument:

Predicted Succeeded and Failed lines:


### Check the completed connections

Run your completed program after the dependency setup succeeds. Record both summary lines and compare them with your prediction. Count the test methods in this completed draft and explain any difference from the earlier four-test boundary program. Use the selected methods to justify the total rather than copying a previous report.

In [ ]:
Your response:

Actual Succeeded and Failed lines:

Comparison with my prediction:

Selected test methods and their total:

Why this total differs from the earlier boundary program:

Any correction I made and the result after rerunning:


<details>
<summary>Show answer</summary>

TEST_MARK is Test, which marks positiveCounts for JUnit discovery. CHECK_EQUAL is assertEquals, which compares independently chosen expected 6 with the actual labelCount result. REPORT_MISSING_EXCEPTION is fail, which reports a test failure if the invalid-input call returns normally. TEST_CLASS is LabelToolsTest; the following .class forms the class literal passed to selectClass. The remaining two @Test methods and the runner stay unchanged. This is the exact three-test canonical program, so the completed answer reports 3 successes and 0 failures after dependency setup.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 0
```

Common error: Replacing the class literal with a String. Reversing the expected and actual assertion roles. Leaving out the missing-exception check. Using the four-test count for a draft containing only three tests.

</details>

### Check that an assertion can report a mismatch

The next complete program is an intact four-test starter for one controlled edit. A **baseline** is the initial result used for comparison. First establish that report, then change a single expectation and restore it. Keeping the application fixed makes the source of the changed test result clear.

In [ ]:
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());


### Establish a comparison report

Run the intact four-test starter immediately above before editing it. Record both summary lines and the number of selected tests. This report is your baseline: the result you will compare with one controlled change.

In [ ]:
Your response:

Intact starter: Succeeded and Failed lines:

Selected test count:

Does the sum of the counts match the intended tests?


### Predict a deliberately incorrect expectation

Change only the expected value in `positiveCounts` from `6` to `7`. Leave the application, inputs, other tests, and runner unchanged. Before rerunning, predict both summary lines and identify the test affected. Explain whether this edit tests the assertion’s ability to report a mismatch or introduces a defect into the application.

In [ ]:
Your response:

Edited assertion and the one value changed:

Expected argument and actual application result:

Predicted result of the affected test:

Predicted Succeeded and Failed lines:

What this controlled edit checks:


Return to your four-test starter above. Apply the planned change there, then rerun the complete Java program so the runner selects the updated class definition. Keep the other tests and the supplied runner together in that work cell.

### Observe the mismatch and restore the test

Run the edited program and record its summary. Compare it with your prediction, identifying the expected and actual values in the affected assertion. Explain how the runner can finish normally while reporting a failed test.

Restore the original expected value `6`, rerun the complete program, and record the restored report. Keep all three reports: intact, edited, and restored.

In [ ]:
Your response:

Edited program: actual Succeeded and Failed lines:

Affected assertion: expected and actual values:

Comparison with my prediction:

Why normal kernel completion does not mean every test passed:

Restored program: actual Succeeded and Failed lines:


<details>
<summary>Show answer</summary>

With expected 7, positiveCounts compares 7 against the correct actual result 6 and fails. The two zero-boundary tests and the required-exception test still succeed, so this four-test suite reports Succeeded: 3 and Failed: 1. The runner records the failed assertion and returns normally; its successful kernel completion does not turn the failed test into a pass. Restoring expected 6 returns the complete suite to 4 successes and 0 failures. This controlled edit checks that the selected test runs and its assertion can report a mismatch, while the later production-defect task checks the suite against a different kind of mistake.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(7, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 1
```

Common error: Changing production multiplication to make an intentionally wrong expectation pass. Assuming one failure must always leave two successes, regardless of the suite size. Stopping after the failing run without restoring the correct expectation. Reading normal kernel completion as a successful JUnit suite.

**Additional test: `Restored expected value 6`.** All four assertions or required-exception checks agree with the unchanged application contract after the intentional expectation error is removed.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

</details>

### Use the contract to repair the application

The supplied faulty program now adds the two counts where the contract requires their product. Its four tests retain their independently chosen expectations. Read the draft without changing it. Predict each test’s result and both summary lines. Explain which inputs expose the arithmetic defect and which behavior follows a different path.

Plan a repair to the application’s return expression. Then copy the complete draft into the empty work cell and repair only that expression. Preserve the test expectations and runner.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups + kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

In [ ]:
Your response:

positiveCounts: faulty result versus required result and test outcome:

zeroGroups: faulty result versus required result and test outcome:

negativeGroups: behavior and test outcome:

zeroKits: faulty result versus required result and test outcome:

Predicted faulty Succeeded and Failed lines:

My proposed application repair and its reason:

Why the test expectations should remain unchanged:


### Verify the application repair

Run your repaired complete program and record both summary lines. Confirm the total matches all four intended tests. Explain how the repair affects the numeric results while preserving the negative-input rule. Distinguish this repair from the earlier exercise that changed a correct assertion’s expected value.

In [ ]:
Your response:

Repaired program: actual Succeeded and Failed lines:

Total executed tests:

Numeric cases that expose the original addition defect:

Why the negative-input test follows a different path:

Why changing expected values to match the faulty application would hide the defect:

How this exercise differs from the assertion-mismatch check:


<details>
<summary>Show answer</summary>

The selected LabelToolsTest class has four @Test methods. positiveCounts checks the known product 2 times 3 against 6. zeroGroups checks that zero groups produce zero labels. negativeGroups succeeds because the application raises IllegalArgumentException with the required message; the line that would report a missing exception is skipped. zeroKits checks the separate zero-kit boundary with groups still positive. All four agree with the production contract, so the runner reports Succeeded: 4 and Failed: 0. The class literal selects the test class; defining the class alone would not execute its tests. The new listener receives this run's counts, and the runner session closes after execute finishes. The expected values come from the behavior contract, not from duplicating the implementation expression.

In the faulty version, addition returns 5 for (2, 3), 3 for (0, 3), and 2 for (2, 0). Those results violate the independently specified product expectations, so the three numeric tests fail. The guard still rejects a negative count with the correct message, so negativeGroups succeeds. The faulty summary is 1 success and 3 failures.

Restoring groups * kits repairs all three numeric behaviors while preserving the rejection rule. The unchanged tests then report 4 successes and 0 failures.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class LabelTools {
    public static int labelCount(int groups, int kits) {
        if (groups < 0 || kits < 0) {
            throw new IllegalArgumentException("Counts must be nonnegative.");
        }
        return groups * kits;
    }
}
class LabelToolsTest {
    public LabelToolsTest() { }
    @Test
    public void positiveCounts() {
        Assertions.assertEquals(6, LabelTools.labelCount(2, 3));
    }
    @Test
    public void zeroGroups() {
        Assertions.assertEquals(0, LabelTools.labelCount(0, 3));
    }
    @Test
    public void negativeGroups() {
        try {
            LabelTools.labelCount(-1, 3);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Counts must be nonnegative.", problem.getMessage());
        }
    }
    @Test
    public void zeroKits() {
        Assertions.assertEquals(0, LabelTools.labelCount(2, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(LabelToolsTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

Common error: Changing the tests to agree with addition even though the contract requires a product. Changing the correct negative-input guard while fixing arithmetic. Assuming a passing rejection test means the numeric behavior is also correct.

</details>

## Independent Practice

### Build tests for workshop seats

Workshop staff need the number of unreserved seats. The supplied `SeatMath.remaining` method receives capacity and reserved seats as whole-number counts. Both counts must be nonnegative, and reservations must fit the capacity. Valid requests return capacity minus reserved seats. Invalid requests raise `IllegalArgumentException` with the message `Reservation must fit the capacity.`

Plan three independently named tests: eight seats with three reserved leaves five; eight with eight reserved leaves zero; eight with nine reserved must raise the required exception and message. The exception test must report failure if the call returns normally.

Write the three public, nonstatic, parameterless `void` methods marked `@Test` in the empty work cell, using the complete starter below. Keep the application and runner unchanged. Give each test its own inputs. Before running, record each expectation and predict both summary lines. Run the dependency setup first in each fresh kernel.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    // Add the three required test methods here.
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

In [ ]:
Your response:

Test1 name, inputs, and independently chosen expected result:

Test2 name, inputs, and independently chosen expected result:

Test3 name, inputs, required exception and exact message:

How test3 detects an unexpectedly normal return:

Class literal selected by the runner:

Predicted Succeeded and Failed lines:


### Check your three behavior tests

Run your complete program and record both summary lines. Check that their sum equals the three intended test methods. Compare each expectation with the supplied seat contract. Explain how your exception test detects an invalid call that unexpectedly returns normally, and why the required application exception can count as a successful test.

In [ ]:
Your response:

Actual Succeeded and Failed lines:

Total tests compared with my intended three methods:

Any mismatch with my prediction and correction made:

Statement that detects an unexpectedly normal invalid-input return:

Why the required exception can be a successful outcome:


### Predict a mismatch in your own test

Starting from your complete three-test baseline, change only the expected `5` in the test for `remaining(8, 3)` to `6`. Keep the application and all other tests unchanged. Before rerunning, predict both summary lines. Identify the expected and actual values and explain what this deliberate edit checks.

In [ ]:
Your response:

Changed test and expected value:

Unchanged actual application result:

Predicted Succeeded and Failed lines:

What the deliberate mismatch checks:


Return to your SeatMathTest work cell above. Apply the planned change there, then rerun the complete Java program so the runner selects the updated class definition. Keep the other tests and the supplied runner together in that work cell.

### Record the failure and restore the baseline

Run the deliberately edited suite. Record its report and explain any difference from your prediction. Explain how a normally completed Java cell can still contain a failed JUnit test. Restore expected `5` and rerun the entire three-test program before adding any boundary tests. Record the restored report separately.

In [ ]:
Your response:

Edited suite: actual Succeeded and Failed lines:

Comparison with my prediction:

Meaning of normal kernel completion versus the test report:

Restored three-test suite: actual Succeeded and Failed lines:


### Add a negative-capacity case

The seat contract rejects a negative capacity even when no seats are reserved. Keep your original three tests and add a public `@Test` method named `negativeCapacity`. Call `SeatMath.remaining(-1, 0)`. Report failure if it returns normally; catch only `IllegalArgumentException` and check the exact message `Reservation must fit the capacity.`

Before running the expanded program, explain the distinct rule this test protects and predict both summary lines and the total number of executed tests.

In [ ]:
Your response:

New input and required behavior:

How the test detects an unexpectedly normal return:

Exact required message:

Predicted Succeeded and Failed lines:

Expected total test count:


Return to your SeatMathTest work cell above. Apply the planned change there, then rerun the complete Java program so the runner selects the updated class definition. Keep the other tests and the supplied runner together in that work cell.

### Check the expanded report

Run the whole program with `negativeCapacity` included. Record both summary lines, add the counts, and compare that total with the intended tests. Explain why a capacity below zero deserves a separate case from reservations that exceed a positive capacity. Retain this test for the next step.

In [ ]:
Your response:

Actual Succeeded and Failed lines:

Expected total and actual sum:

Comparison with my prediction:

Distinct behavior protected by negativeCapacity:

Any correction and rerun result:


### Add a negative-reservation case

Keep the existing tests and add a public `@Test` method named `negativeReservations`. Use `SeatMath.remaining(8, -1)`. The contract requires `IllegalArgumentException` and the message `Reservation must fit the capacity.` Include the assertion that fails if the call returns normally, and catch only the required exception type.

Explain what new input rule this case checks. Predict both summary lines and the updated total before running the complete program.

In [ ]:
Your response:

Input and distinct required behavior:

Required exception and message:

How an unexpectedly normal return is detected:

Predicted Succeeded and Failed lines:

Expected total test count:


Return to your SeatMathTest work cell above. Apply the planned change there, then rerun the complete Java program so the runner selects the updated class definition. Keep the other tests and the supplied runner together in that work cell.

### Compare the two invalid-input rules

Run the suite with both added exception tests. Record its summary and total, then compare them with your prediction. Explain why the negative-capacity case cannot replace the negative-reservations case: identify which input is invalid in each. Keep both tests for the final addition.

In [ ]:
Your response:

Actual Succeeded and Failed lines:

Expected total and actual sum:

Comparison with my prediction:

Why each negative-input case needs its own check:

Any correction and rerun result:


### Test the empty-workshop boundary

Keep the five existing tests. Add a public `@Test` method named `zeroCapacityAndReservations` that compares expected `0` with `SeatMath.remaining(0, 0)`. Explain how this boundary follows the stated nonnegative-count rule. Predict both summary lines and the new total before running the complete program.

In [ ]:
Your response:

Boundary inputs and expected behavior:

Why the expectation follows from the contract:

Predicted Succeeded and Failed lines:

Expected total test count:


Return to your SeatMathTest work cell above. Apply the planned change there, then rerun the complete Java program so the runner selects the updated class definition. Keep the other tests and the supplied runner together in that work cell.

### Review the complete set of behaviors

Run the complete suite and record both summary lines and their sum. Compare the total with all six intended tests. Explain what the zero-capacity case adds to the previous cases. If a result differs from the contract, identify the mismatch, repair your tests as needed, and rerun the affected suite. Keep the supplied application unchanged.

Explain why these tests describe observable requirements rather than checking that a particular source line exists. Retain the deliberate-failure report, restored baseline, and each expanded-suite report.

In [ ]:
Your response:

Final actual Succeeded and Failed lines:

Expected total and actual sum:

Behavior added by zeroCapacityAndReservations:

Any mismatch, repair, and rerun result:

Why these checks describe required behavior rather than implementation lines:


<details>
<summary>Show answer</summary>

The provided SeatMath rejects negative capacity, negative reservations and reservations beyond capacity. someSeatsAvailable checks a normal remaining value of 5. allSeatsReserved checks the zero-remaining boundary. tooManyReservations requires IllegalArgumentException with the exact contract message; Assertions.fail makes an unexpected normal return fail the test instead of slipping through. Each test supplies its own input values. The runner selects SeatMathTest.class and reports 3 successes with 0 failures for the complete baseline. Its imports and runner are included, but the pinned dependency must first be loaded into each new kernel.

Changing expected 5 to 6 makes only someSeatsAvailable fail, so the three-test suite reports 2 successes and 1 failure while the runner still returns normally. Restoring 5 produces 3 successes and 0 failures.

Adding negativeCapacity gives four tests; adding negativeReservations gives five; adding zeroCapacityAndReservations gives six. All added expectations follow the fixed contract, so the successive correct summaries are 4/0, 5/0 and 6/0. The two negative-input tests check both the required exception and its message, including the case where no exception occurs. The zero/zero case confirms that a capacity of zero is valid when no seats are reserved. Keeping the earlier tests means the original behaviors remain covered as the suite grows.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 3
Failed: 0
```

Common error: Editing the supplied production method to match a faulty test. Catching a broad failure instead of the required IllegalArgumentException. Leaving out Assertions.fail after the call expected to throw. Selecting the previous LabelToolsTest class instead of SeatMathTest. Counting zero failures as enough when the starter still has no completed tests.

**Additional test: Deliberately wrong baseline expectation: 5 becomes 6.** The actual normal result remains 5. Only that assertion fails, leaving two successes and one failure; this is an intended observed failure, not the final solution.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(6, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 2
Failed: 1
```

**Additional test: Baseline plus negativeCapacity.** The new invalid-capacity test includes its own call, missing-exception safeguard and required message check. The original three tests remain, so four tests pass.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 4
Failed: 0
```

**Additional test: Also add negativeReservations.** The additional test checks a negative reservation count independently of test execution order. Keeping all earlier tests increases the passing count to five.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeReservations() {
        try {
            SeatMath.remaining(8, -1);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 5
Failed: 0
```

**Additional test: Also add zeroCapacityAndReservations.** Zero capacity and zero reservations produce zero remaining seats. This sixth distinct test preserves valid boundary behavior alongside the three baseline and two negative-input tests.

```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.Assertions;
import org.junit.platform.engine.discovery.DiscoverySelectors;
import org.junit.platform.launcher.Launcher;
import org.junit.platform.launcher.LauncherSession;
import org.junit.platform.launcher.LauncherDiscoveryRequest;
import org.junit.platform.launcher.core.LauncherDiscoveryRequestBuilder;
import org.junit.platform.launcher.core.LauncherFactory;
import org.junit.platform.launcher.listeners.SummaryGeneratingListener;
class SeatMath {
    public static int remaining(int capacity, int reserved) {
        if (capacity < 0 || reserved < 0 || reserved > capacity) {
            throw new IllegalArgumentException("Reservation must fit the capacity.");
        }
        return capacity - reserved;
    }
}
class SeatMathTest {
    public SeatMathTest() { }
    @Test
    public void someSeatsAvailable() {
        Assertions.assertEquals(5, SeatMath.remaining(8, 3));
    }
    @Test
    public void allSeatsReserved() {
        Assertions.assertEquals(0, SeatMath.remaining(8, 8));
    }
    @Test
    public void tooManyReservations() {
        try {
            SeatMath.remaining(8, 9);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeCapacity() {
        try {
            SeatMath.remaining(-1, 0);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void negativeReservations() {
        try {
            SeatMath.remaining(8, -1);
            Assertions.fail("Expected IllegalArgumentException");
        } catch (IllegalArgumentException problem) {
            Assertions.assertEquals("Reservation must fit the capacity.", problem.getMessage());
        }
    }
    @Test
    public void zeroCapacityAndReservations() {
        Assertions.assertEquals(0, SeatMath.remaining(0, 0));
    }
}
LauncherDiscoveryRequest request = LauncherDiscoveryRequestBuilder.request()
    .selectors(DiscoverySelectors.selectClass(SeatMathTest.class)).build();
SummaryGeneratingListener listener = new SummaryGeneratingListener();
try (LauncherSession session = LauncherFactory.openSession()) {
    Launcher launcher = session.getLauncher();
    launcher.registerTestExecutionListeners(listener);
    launcher.execute(request);
}
System.out.println("Succeeded: " + listener.getSummary().getTestsSucceededCount());
System.out.println("Failed: " + listener.getSummary().getTestsFailedCount());
```

Expected output:

```text
Succeeded: 6
Failed: 0
```

</details>

## Summary

### Retrieve the testing safeguards

Without reopening the reading or answers, explain why the required-exception test calls `Assertions.fail` after the application call. Describe the incorrect behavior that could otherwise pass unnoticed. Then explain why a report of zero failed tests is incomplete evidence unless you also check how many intended tests executed.

In [ ]:
Your response:

Job of Assertions.fail after the application call:

Incorrect behavior an incomplete exception test could miss:

Why zero failures alone is insufficient:

How I check that the intended tests executed:


<details><summary>Show answer: require the behavior and check the count</summary>

`Assertions.fail` reports a failed test when the application call unexpectedly returns normally. Without that line, an invalid input could return a number, skip the specific `catch`, and reach the end of the test without any assertion detecting the missing exception.

```java
LabelTools.labelCount(-1, 3);
Assertions.fail("Expected IllegalArgumentException");
```

These are the two statements inside the test's `try` body. A correct application exception transfers control to the handler before the second line. An incorrect normal return reaches the failure assertion. The handler then has a separate job on the valid exceptional path: check the required message.

For the final six-test seat suite, the intended report is:

```text
Succeeded: 6
Failed: 0
```

Zero failures alone could describe a run that discovered no tests. In these examples, no tests are skipped or aborted, so checking the sum against the intended count helps establish that the selected cases ran. The count still cannot prove that the assertions check every requirement; the deliberately incomplete exception test showed why.

Common errors include treating an expected application exception as a failed test, omitting the safeguard after an unexpectedly normal return, and checking only the failed count.

</details>

Tests turn method contracts into repeatable evidence. Independent expectations, separate boundary cases, and complete exception checks make that evidence more useful. An assertion sanity check confirms that a mismatch is reported. Retained regression tests help detect behavior broken by later application changes.

## Reflection

Choose a method from your earlier Java practice and state its input/result contract. Propose one normal case, one boundary case, and one invalid-input case to preserve as automated tests. Give each case concrete inputs and an expected result or required exception. Name a relevant implementation mistake each test could detect.

Explain how retaining these checks would help you notice when a later change breaks previously correct behavior.

In [ ]:
Your response:

Chosen method and its input/result contract:

Normal case: inputs, expected behavior, and mistake detected:

Boundary case: inputs, expected behavior, and mistake detected:

Invalid-input case: inputs, required behavior, and mistake detected:

How retained tests would help detect a later regression:


Exception handling describes how a program responds when work fails. Cleanup ensures that resources are released as control leaves the protected work. Automated tests make selected expectations repeatable after changes. You will bring these ideas together when reading and writing text files: handle invalid data deliberately, release opened resources, and preserve required behavior with tests.

## Supplemental Reading

- [JUnit 5.13.4 assertions](https://docs.junit.org/5.13.4/user-guide/index.html#writing-tests-assertions) introduces Jupiter behavior checks.
- [JUnit Platform Launcher API](https://docs.junit.org/5.13.4/user-guide/index.html#launcher-api) explains test selection, execution, and listeners.
- [JUnit 5.13.4 Assertions API](https://docs.junit.org/5.13.4/api/org.junit.jupiter.api/org/junit/jupiter/api/Assertions.html) documents assertEquals and fail.
- [Original IJava dependency magics](https://github.com/SpencerPark/IJava/blob/master/docs/magics.md) explains kernel dependency setup.